# Average of each animal variable across THI values

For every integer THI value we compute the **mean of each animal (production / welfare) variable** and plot it against THI.

- THI used: `THI_1_avg` (the test-day THI; consistent with the climate-signal methodology).
- Animal variables: milk yield, ECM, fat %, protein %, somatic cells / SCS, DIM, parity, AFC.
- Data source: `Final_Merged_Data.csv` (test-day records merged with weather indices).

This is a **read-only** analysis notebook — it does not modify any existing file or dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = '../Thesis_Data/Final_Data/Final_Merged_Data.csv'
THI_COL   = 'THI_1_avg'   # test-day THI

# Candidate animal variables (only those present + non-constant are kept below)
ANIMAL_VARS = ['milk_kg', 'ECM', 'fat_p', 'protein_p', 'cells', 'SCS', 'DIM', 'parity', 'AFC']

# Readable labels for the plots
LABELS = {
    'milk_kg':   'Milk yield (kg)',
    'ECM':       'Energy-corrected milk (kg)',
    'fat_p':     'Fat %',
    'protein_p': 'Protein %',
    'cells':     'Somatic cell count (x1000/mL)',
    'SCS':       'Somatic cell score',
    'DIM':       'Days in milk',
    'parity':    'Parity',
    'AFC':       'Age at first calving',
}

In [ ]:
# Load only the columns we need (file is large)
usecols = [THI_COL] + ANIMAL_VARS
df = pd.read_csv(DATA_PATH, usecols=lambda c: c in usecols)

# Keep only variables that are actually present and vary (drop constants such as NM)
vars_present = [v for v in ANIMAL_VARS if v in df.columns and df[v].nunique(dropna=True) > 1]
dropped = [v for v in ANIMAL_VARS if v not in vars_present]

print(f'Records loaded: {len(df):,}')
print(f'THI range     : {df[THI_COL].min():.1f} to {df[THI_COL].max():.1f}')
print(f'Animal vars   : {vars_present}')
if dropped:
    print(f'Dropped (missing/constant): {dropped}')

In [ ]:
# Bin THI to integer values and average each animal variable per THI value
df = df.dropna(subset=[THI_COL]).copy()
df['THI_bin'] = df[THI_COL].round().astype(int)

grp = df.groupby('THI_bin')
means  = grp[vars_present].mean()
counts = grp.size().rename('n')

# Ignore sparsely-populated THI values (unstable means)
MIN_N = 30
valid = counts[counts >= MIN_N].index
means  = means.loc[valid]
counts = counts.loc[valid]

summary = means.join(counts)
summary.head(10)

In [ ]:
# One subplot per animal variable: mean(variable) vs THI
n = len(vars_present)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows), squeeze=False)
axes = axes.ravel()

for ax, var in zip(axes, vars_present):
    ax.plot(means.index, means[var], marker='o', ms=4, lw=1.5, color='#c0392b')
    ax.set_title(LABELS.get(var, var))
    ax.set_xlabel('THI (THI_1_avg, rounded)')
    ax.set_ylabel('Mean')
    ax.grid(alpha=0.3)

# Hide any unused axes
for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle('Average of each animal variable across THI values', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Context: how many test-day records fall at each THI value
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(counts.index, counts.values, color='#2c3e50', width=0.9)
ax.set_title(f'Number of records per THI value (min {MIN_N} to be plotted)')
ax.set_xlabel('THI (THI_1_avg, rounded)')
ax.set_ylabel('Record count')
ax.grid(alpha=0.3, axis='y')
fig.tight_layout()
plt.show()